# 从零实现 SAM 风格可提示分割：点、框、双向 decoder 与多 mask

本 Notebook 实现的是教学版 **SAM 风格协议**，不是 Meta SAM 的官方结构复刻，也不导入任何 SAM 包。我们手写 image encoder、正/负/填充点与 box prompt encoder、mask token、双向 cross-attention decoder、multi-mask 输出和 IoU quality head，重点验证提示语义、padding 与候选选择。

合成图像中同时出现左右两个目标，提示决定要分割哪一个。受控训练只说明实现能学习这个小规则，不代表开放世界、零样本或真实高分辨率分割能力。

In [ ]:
import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import warnings  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED59=5901  # 计算并保存当前步骤的中间状态。
random.seed(SEED59); torch.manual_seed(SEED59); torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE59=torch.device("cpu")  # 计算并保存当前步骤的中间状态。
def canonical_json59(value): return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
assert DEVICE59.type=="cpu" and torch.get_num_threads()==1  # 用受控断言验证关键不变量。

## 1. 坐标与 prompt 合同

点坐标采用 `(x,y)` 像素坐标，并按 `(x/(W-1), y/(H-1))` 归一化；label 为 `1=positive`、`0=negative`、`-1=padding`。box 为闭区间角点 `(x0,y0,x1,y1)` 且必须有正面积。padding token 不允许参与 key/value attention；每个样本至少有一个有效点或 box，空 prompt 直接拒绝，避免悄悄退化为无提示语义分割。

In [ ]:
def normalize_coords59(coords,height,width):  # 定义本节可复用的核心函数。
    if coords.shape[-1]!=2 or height<2 or width<2 or not torch.isfinite(coords).all(): raise ValueError("invalid_coordinates")  # 按当前条件选择后续控制路径。
    if (coords[...,0]<0).any() or (coords[...,0]>width-1).any() or (coords[...,1]<0).any() or (coords[...,1]>height-1).any():  # 按当前条件选择后续控制路径。
        raise ValueError("coordinates_outside_viewport")  # 遇到非法合同立即显式失败。
    scale=torch.tensor([width-1,height-1],dtype=coords.dtype,device=coords.device)  # 计算并保存当前步骤的中间状态。
    return coords/scale  # 返回当前分支计算出的结果。

class PointBoxPromptEncoder59(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,dim=24):  # 定义本节可复用的核心函数。
        super().__init__(); self.dim=dim  # 计算并保存当前步骤的中间状态。
        self.coord=nn.Sequential(nn.Linear(2,dim),nn.GELU(),nn.Linear(dim,dim))  # 计算并保存当前步骤的中间状态。
        self.point_label=nn.Embedding(3,dim); self.corner_type=nn.Embedding(2,dim)  # 计算并保存当前步骤的中间状态。
    def forward(self,points,labels,boxes,image_hw=(16,16),box_mask=None):  # 定义本节可复用的核心函数。
        if points.ndim!=3 or labels.shape!=points.shape[:2] or points.shape[-1]!=2: raise ValueError("point_shape_contract")  # 按当前条件选择后续控制路径。
        if labels.dtype!=torch.long or not torch.isin(labels,torch.tensor([-1,0,1],device=labels.device)).all(): raise ValueError("point_label_contract")  # 按当前条件选择后续控制路径。
        b=points.shape[0]; h,w=image_hw; valid=labels!=-1  # 计算并保存当前步骤的中间状态。
        safe_points=torch.where(valid[...,None],points,torch.zeros_like(points))  # 计算并保存当前步骤的中间状态。
        point_norm=normalize_coords59(safe_points,h,w)  # 计算并保存当前步骤的中间状态。
        point_tokens=self.coord(point_norm)+self.point_label((labels+1).clamp(0,2))  # 计算并保存当前步骤的中间状态。
        point_tokens=torch.where(valid[...,None],point_tokens,torch.zeros_like(point_tokens)); padding=~valid  # 计算并保存当前步骤的中间状态。
        if boxes is None:  # 按当前条件选择后续控制路径。
            if box_mask is not None: raise ValueError("box_mask_without_boxes")  # 按当前条件选择后续控制路径。
            boxes=points.new_empty((b,0,4))  # 计算并保存当前步骤的中间状态。
        if boxes.ndim!=3 or boxes.shape[0]!=b or boxes.shape[-1]!=4: raise ValueError("box_shape_contract")  # 按当前条件选择后续控制路径。
        if box_mask is None: box_mask=torch.ones(boxes.shape[:2],dtype=torch.bool,device=points.device)  # 按当前条件选择后续控制路径。
        if box_mask.shape!=boxes.shape[:2] or box_mask.dtype!=torch.bool: raise ValueError("box_mask_contract")  # 按当前条件选择后续控制路径。
        if boxes.shape[1]:  # 按当前条件选择后续控制路径。
            valid_boxes=boxes[box_mask]  # 计算并保存当前步骤的中间状态。
            if valid_boxes.numel() and (not torch.isfinite(valid_boxes).all() or (valid_boxes[:,2]<=valid_boxes[:,0]).any() or (valid_boxes[:,3]<=valid_boxes[:,1]).any()): raise ValueError("invalid_box")  # 按当前条件选择后续控制路径。
            safe_boxes=torch.where(box_mask[...,None],boxes,torch.zeros_like(boxes))  # 计算并保存当前步骤的中间状态。
            corners=torch.stack([safe_boxes[...,:2],safe_boxes[...,2:]],dim=2)  # 计算并保存当前步骤的中间状态。
            corner_norm=normalize_coords59(corners,h,w)  # 计算并保存当前步骤的中间状态。
            box_tokens=self.coord(corner_norm)+self.corner_type.weight[None,None]  # 计算并保存当前步骤的中间状态。
            box_tokens=box_tokens.reshape(b,-1,self.dim); box_padding=(~box_mask)[...,None].expand(-1,-1,2).reshape(b,-1)  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            box_tokens=points.new_empty((b,0,self.dim)); box_padding=torch.empty(b,0,dtype=torch.bool,device=points.device)  # 计算并保存当前步骤的中间状态。
        tokens=torch.cat([point_tokens,box_tokens],1); padding=torch.cat([padding,box_padding],1)  # 计算并保存当前步骤的中间状态。
        if padding.all(1).any(): raise ValueError("empty_prompt_is_not_allowed")  # 按当前条件选择后续控制路径。
        return tokens,padding  # 返回当前分支计算出的结果。

coord_oracle59=normalize_coords59(torch.tensor([[[0.,0.],[15.,15.],[7.5,3.75]]]),16,16)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(coord_oracle59[0,0],torch.tensor([0.,0.])) and torch.allclose(coord_oracle59[0,1],torch.tensor([1.,1.]))  # 用受控断言验证关键不变量。
assert torch.allclose(coord_oracle59[0,2],torch.tensor([.5,.25]))  # 用受控断言验证关键不变量。
prompt_probe59=PointBoxPromptEncoder59()  # 计算并保存当前步骤的中间状态。
tok59,pad59=prompt_probe59(torch.tensor([[[3.,7.],[12.,7.],[0.,0.]]]),torch.tensor([[1,0,-1]]),torch.tensor([[[2.,4.,6.,12.]]]))  # 计算并保存当前步骤的中间状态。
assert tok59.shape==(1,5,24) and torch.equal(pad59,torch.tensor([[False,False,True,False,False]]))  # 用受控断言验证关键不变量。
same_xy_tok59,_=prompt_probe59(torch.tensor([[[3.,7.]],[[3.,7.]]]),torch.tensor([[1],[0]]),None)  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(same_xy_tok59[0,0],same_xy_tok59[1,0])  # 同一坐标的正/负点必须有不同语义
try:  # 尝试执行可能失败的受控操作。
    prompt_probe59(torch.zeros(1,2,2),torch.full((1,2),-1,dtype=torch.long),None)  # 计算并保存当前步骤的中间状态。
    raise AssertionError("empty prompt should fail")  # 遇到非法合同立即显式失败。
except ValueError as exc: assert "empty_prompt" in str(exc)  # 捕获预期异常并验证失败分支。

## 2. 手写 image encoder 与空间 token

两层步长卷积把 `[B,1,16,16]` 编成 `[B,16,D]`。纯卷积特征不足以让提示坐标与位置对齐，因此向每个 4×4 网格 token 加入归一化二维坐标的可学习投影。这里没有调用 `torchvision.models`、`timm`、SAM 或现成 Transformer；真实 SAM 使用更强大且预训练过的图像编码器。

In [ ]:
class ImageEncoder59(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,dim=24):  # 定义本节可复用的核心函数。
        super().__init__(); self.dim=dim  # 计算并保存当前步骤的中间状态。
        self.stem=nn.Sequential(nn.Conv2d(1,dim,3,stride=2,padding=1),nn.GELU(),nn.Conv2d(dim,dim,3,stride=2,padding=1),nn.GELU())  # 计算并保存当前步骤的中间状态。
        self.pos=nn.Linear(2,dim)  # 计算并保存当前步骤的中间状态。
    def forward(self,images):  # 定义本节可复用的核心函数。
        if images.ndim!=4 or images.shape[1:]!=(1,16,16) or not torch.isfinite(images).all(): raise ValueError("image_contract")  # 按当前条件选择后续控制路径。
        feat=self.stem(images); b,d,h,w=feat.shape  # 计算并保存当前步骤的中间状态。
        yy,xx=torch.meshgrid(torch.linspace(0,1,h,device=feat.device),torch.linspace(0,1,w,device=feat.device),indexing="ij")  # 计算并保存当前步骤的中间状态。
        pos=self.pos(torch.stack([xx,yy],-1).reshape(1,h*w,2))  # 计算并保存当前步骤的中间状态。
        return feat.flatten(2).transpose(1,2)+pos,(h,w)  # 返回当前分支计算出的结果。

image_encoder_probe59=ImageEncoder59(); image_tokens59,hw59=image_encoder_probe59(torch.zeros(2,1,16,16))  # 计算并保存当前步骤的中间状态。
assert image_tokens59.shape==(2,16,24) and hw59==(4,4)  # 用受控断言验证关键不变量。
assert not torch.allclose(image_tokens59[:,0],image_tokens59[:,-1])  # 用受控断言验证关键不变量。

## 3. 手写 cross-attention 与双向更新

`ManualAttention59(q, kv)` 显式实现多头 $QK^T/\sqrt{d_h}$。decoder 先让 prompt/mask token 查询 image，再做 token self-attention，最后让 image 查询 prompt；这就是教学版 two-way 信息交换。key padding 在 softmax 前写入 `-inf`，并拒绝“某样本全部 key 被遮蔽”，防止产生 NaN。

In [ ]:
class ManualAttention59(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,dim=24,heads=4):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if dim%heads: raise ValueError("dim_not_divisible_by_heads")  # 按当前条件选择后续控制路径。
        self.dim,self.heads,self.hd=dim,heads,dim//heads  # 计算并保存当前步骤的中间状态。
        self.q=nn.Linear(dim,dim); self.k=nn.Linear(dim,dim); self.v=nn.Linear(dim,dim); self.out=nn.Linear(dim,dim)  # 计算并保存当前步骤的中间状态。
    def forward(self,q_input,kv_input,kv_padding=None):  # 定义本节可复用的核心函数。
        if q_input.ndim!=3 or kv_input.ndim!=3 or q_input.shape[0]!=kv_input.shape[0]: raise ValueError("attention_shape")  # 按当前条件选择后续控制路径。
        b,nq,_=q_input.shape; nk=kv_input.shape[1]  # 计算并保存当前步骤的中间状态。
        if kv_padding is not None:  # 按当前条件选择后续控制路径。
            if kv_padding.shape!=(b,nk) or kv_padding.dtype!=torch.bool or kv_padding.all(1).any(): raise ValueError("invalid_key_padding")  # 按当前条件选择后续控制路径。
        q=self.q(q_input).reshape(b,nq,self.heads,self.hd).transpose(1,2)  # 计算并保存当前步骤的中间状态。
        k=self.k(kv_input).reshape(b,nk,self.heads,self.hd).transpose(1,2)  # 计算并保存当前步骤的中间状态。
        v=self.v(kv_input).reshape(b,nk,self.heads,self.hd).transpose(1,2)  # 计算并保存当前步骤的中间状态。
        scores=q@k.transpose(-2,-1)/math.sqrt(self.hd)  # 计算并保存当前步骤的中间状态。
        if kv_padding is not None: scores=scores.masked_fill(kv_padding[:,None,None,:],float("-inf"))  # 按当前条件选择后续控制路径。
        weights=scores.softmax(-1); out=(weights@v).transpose(1,2).reshape(b,nq,self.dim)  # 计算并保存当前步骤的中间状态。
        return self.out(out),weights  # 返回当前分支计算出的结果。

q59=torch.randn(2,3,24); kv59=torch.randn(2,5,24); p59=torch.tensor([[False,False,False,True,True],[False]*5])  # 计算并保存当前步骤的中间状态。
a59,w59=ManualAttention59()(q59,kv59,p59)  # 计算并保存当前步骤的中间状态。
assert a59.shape==(2,3,24) and torch.allclose(w59.sum(-1),torch.ones(2,4,3),atol=1e-6)  # 用受控断言验证关键不变量。
assert w59[0,...,3:].abs().max()==0 and torch.isfinite(a59).all()  # 用受控断言验证关键不变量。

## 4. mask token、多候选 mask 与 IoU head

三个可学习 mask token 生成三个动态通道向量，与更新后的 image embedding 点积得到低分辨率 mask，再上采样回 16×16。IoU head 只预测每个候选的质量，不能用它替代真实 mask 监督。候选维 shape 为 `[B,K,H,W]`；推理根据预测 IoU 选择，而不是偷偷查看 ground truth。

In [ ]:
class TwoWayMaskDecoder59(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,dim=24,heads=4,num_masks=3):  # 定义本节可复用的核心函数。
        super().__init__(); self.dim,self.num_masks=dim,num_masks  # 计算并保存当前步骤的中间状态。
        self.mask_tokens=nn.Parameter(torch.randn(1,num_masks,dim)*.02)  # 计算并保存当前步骤的中间状态。
        self.p_to_i=ManualAttention59(dim,heads); self.self_attn=ManualAttention59(dim,heads); self.i_to_p=ManualAttention59(dim,heads)  # 计算并保存当前步骤的中间状态。
        self.np=nn.LayerNorm(dim); self.ni=nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.hyper=nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.Linear(dim,dim))  # 计算并保存当前步骤的中间状态。
        self.iou=nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.Linear(dim,1))  # 计算并保存当前步骤的中间状态。
    def forward(self,image_tokens,image_hw,prompt_tokens,prompt_padding,output_hw=(16,16)):  # 定义本节可复用的核心函数。
        b=image_tokens.shape[0]; masks=self.mask_tokens.expand(b,-1,-1)  # 计算并保存当前步骤的中间状态。
        tokens=torch.cat([masks,prompt_tokens],1)  # 计算并保存当前步骤的中间状态。
        token_padding=torch.cat([torch.zeros(b,self.num_masks,dtype=torch.bool,device=tokens.device),prompt_padding],1)  # 计算并保存当前步骤的中间状态。
        delta,_=self.p_to_i(self.np(tokens),self.ni(image_tokens)); tokens=tokens+delta  # 计算并保存当前步骤的中间状态。
        delta,_=self.self_attn(self.np(tokens),self.np(tokens),token_padding); tokens=tokens+delta  # 计算并保存当前步骤的中间状态。
        delta,_=self.i_to_p(self.ni(image_tokens),self.np(tokens),token_padding); image_tokens=image_tokens+delta  # 计算并保存当前步骤的中间状态。
        h,w=image_hw; feature=image_tokens.transpose(1,2).reshape(b,self.dim,h,w)  # 计算并保存当前步骤的中间状态。
        dynamic=self.hyper(tokens[:,:self.num_masks]); low=torch.einsum("bkd,bdhw->bkhw",dynamic,feature)  # 计算并保存当前步骤的中间状态。
        logits=F.interpolate(low,size=output_hw,mode="nearest")  # 计算并保存当前步骤的中间状态。
        return logits,self.iou(tokens[:,:self.num_masks]).squeeze(-1)  # 返回当前分支计算出的结果。

class TinyPromptSegmenter59(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,dim=24,heads=4,num_masks=3):  # 定义本节可复用的核心函数。
        super().__init__(); self.image=ImageEncoder59(dim); self.prompt=PointBoxPromptEncoder59(dim); self.decoder=TwoWayMaskDecoder59(dim,heads,num_masks)  # 计算并保存当前步骤的中间状态。
    def forward(self,images,points,labels,boxes,box_mask=None):  # 定义本节可复用的核心函数。
        image_tokens,hw=self.image(images); prompt_tokens,pad=self.prompt(points,labels,boxes,images.shape[-2:],box_mask)  # 计算并保存当前步骤的中间状态。
        return self.decoder(image_tokens,hw,prompt_tokens,pad,images.shape[-2:])  # 返回当前分支计算出的结果。

segmenter_probe59=TinyPromptSegmenter59()  # 计算并保存当前步骤的中间状态。
img_probe59=torch.zeros(1,1,16,16); pts_probe59=torch.tensor([[[3.,7.],[12.,7.],[0.,0.]]]); lab_probe59=torch.tensor([[1,0,-1]])  # 计算并保存当前步骤的中间状态。
box_probe59=torch.tensor([[[2.,4.,6.,12.]]])  # 计算并保存当前步骤的中间状态。
logits_probe59,iou_probe59=segmenter_probe59(img_probe59,pts_probe59,lab_probe59,box_probe59)  # 计算并保存当前步骤的中间状态。
assert logits_probe59.shape==(1,3,16,16) and iou_probe59.shape==(1,3)  # 用受控断言验证关键不变量。
segmenter_probe59.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    point_only59,_=segmenter_probe59(img_probe59,pts_probe59[:,:2],lab_probe59[:,:2],None)  # 计算并保存当前步骤的中间状态。
    box_only59,_=segmenter_probe59(img_probe59,torch.zeros(1,1,2),torch.full((1,1),-1,dtype=torch.long),box_probe59)  # 计算并保存当前步骤的中间状态。
    moved_box59,_=segmenter_probe59(img_probe59,torch.zeros(1,1,2),torch.full((1,1),-1,dtype=torch.long),torch.tensor([[[9.,4.,13.,12.]]]))  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(point_only59,box_only59) and not torch.allclose(box_only59,moved_box59)  # 用受控断言验证关键不变量。
mixed_box_mask59=torch.tensor([[False],[True]])  # 计算并保存当前步骤的中间状态。
mixed_logits59,_=segmenter_probe59(img_probe59.expand(2,-1,-1,-1),torch.tensor([[[3.,7.]],[[0.,0.]]]),torch.tensor([[1],[-1]]),box_probe59.expand(2,-1,-1),mixed_box_mask59)  # 计算并保存当前步骤的中间状态。
assert mixed_logits59.shape==(2,3,16,16)  # 第一个样本 point-only，第二个样本 box-only

## 5. Prompt 顺序与 padding oracle

点集合的语义不应因调用方排列顺序而变化。本 decoder 没有给 prompt 序号加位置编码，所以在 eval 模式下，交换点及其 label 只会置换辅助 prompt token，不会改变固定 mask token 的结果。追加 padding 点也必须完全无影响；若忘记把 padding 传入 image→prompt attention，这个测试会失败。

In [ ]:
segmenter_probe59.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    base_logits59,base_iou59=segmenter_probe59(img_probe59,pts_probe59[:,:2],lab_probe59[:,:2],box_probe59)  # 计算并保存当前步骤的中间状态。
    rev_logits59,rev_iou59=segmenter_probe59(img_probe59,pts_probe59[:,:2].flip(1),lab_probe59[:,:2].flip(1),box_probe59)  # 计算并保存当前步骤的中间状态。
    pad_logits59,pad_iou59=segmenter_probe59(img_probe59,pts_probe59,lab_probe59,box_probe59)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(base_logits59,rev_logits59,atol=2e-6) and torch.allclose(base_iou59,rev_iou59,atol=2e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(base_logits59,pad_logits59,atol=2e-6) and torch.allclose(base_iou59,pad_iou59,atol=2e-6)  # 用受控断言验证关键不变量。
no_box_tokens59,no_box_pad59=prompt_probe59(pts_probe59[:,:2],lab_probe59[:,:2],None)  # 计算并保存当前步骤的中间状态。
assert tok59.shape[1]==no_box_tokens59.shape[1]+3  # 一个 padding 点 + box 两角
assert (~pad59).sum()==4 and (~no_box_pad59).sum()==2  # 用受控断言验证关键不变量。

## 6. 分割 loss、真实 IoU target 与最佳候选选择

每个候选计算 BCE + soft Dice，但分割项只反传每个样本的最优候选（min-over-K），避免把所有 mask token 强迫到同一个 GT 而结构性坍缩；quality head 仍用全部候选的真实 IoU target。target 由 `sigmoid(logit)>0.5` 的离散 mask 与 GT 计算并 `detach`。空集双方都空时 IoU=1。推理只能 `argmax(predicted_iou)`；评估若按真实 IoU 选候选会造成 oracle leakage。

In [ ]:
def binary_iou_targets59(logits,target):  # 定义本节可复用的核心函数。
    if logits.ndim!=4 or target.shape!=(logits.shape[0],logits.shape[2],logits.shape[3]): raise ValueError("iou_shape_contract")  # 按当前条件选择后续控制路径。
    pred=logits.detach().sigmoid()>.5; truth=target[:,None].bool()  # 计算并保存当前步骤的中间状态。
    inter=(pred&truth).sum((-2,-1)).float(); union=(pred|truth).sum((-2,-1)).float()  # 计算并保存当前步骤的中间状态。
    return torch.where(union>0,inter/union,torch.ones_like(union))  # 返回当前分支计算出的结果。

def prompt_loss59(logits,pred_iou,target):  # 定义本节可复用的核心函数。
    target_k=target[:,None].expand_as(logits)  # 计算并保存当前步骤的中间状态。
    bce=F.binary_cross_entropy_with_logits(logits,target_k,reduction="none").mean((-2,-1))  # 计算并保存当前步骤的中间状态。
    probs=logits.sigmoid(); inter=(probs*target_k).sum((-2,-1)); dice=1-(2*inter+1)/(probs.sum((-2,-1))+target_k.sum((-2,-1))+1)  # 计算并保存当前步骤的中间状态。
    iou_target=binary_iou_targets59(logits,target)  # 计算并保存当前步骤的中间状态。
    per_candidate=bce+dice  # 计算并保存当前步骤的中间状态。
    return per_candidate.min(1).values.mean()+.4*F.mse_loss(pred_iou.sigmoid(),iou_target),iou_target  # 返回当前分支计算出的结果。

def select_best59(logits,pred_iou):  # 定义本节可复用的核心函数。
    if pred_iou.shape!=logits.shape[:2]: raise ValueError("candidate_quality_shape")  # 按当前条件选择后续控制路径。
    idx=pred_iou.argmax(1); return logits[torch.arange(len(logits),device=logits.device),idx],idx  # 计算并保存当前步骤的中间状态。

exact_logits59=torch.full((1,3,2,2),-9.); exact_logits59[0,0,0,0]=9; exact_logits59[0,1,0,:]=9  # 计算并保存当前步骤的中间状态。
exact_target59=torch.tensor([[[1.,0.],[0.,0.]]]); targets59=binary_iou_targets59(exact_logits59,exact_target59)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(targets59[0,:2],torch.tensor([1.,.5]))  # 用受控断言验证关键不变量。
chosen59,chosen_idx59=select_best59(exact_logits59,torch.tensor([[.2,.9,.1]]))  # 计算并保存当前步骤的中间状态。
assert chosen_idx59.item()==1 and torch.equal(chosen59,exact_logits59[:,1])  # 用受控断言验证关键不变量。

## 7. 受控提示数据：同一图像、不同目标

每张图都含左右两个亮矩形；奇偶样本分别以左/右矩形为 GT，同时提供目标内正点、另一目标内负点与目标 box。因为图像内容相同，模型不能只靠 image 猜标签。train/validation/test 按样本 ID 切分且三部分都覆盖两种提示语义；真实任务还必须按原图或视频分组，防止同源图泄漏。

In [ ]:
def make_prompt_data59(count=32):  # 定义本节可复用的核心函数。
    g=torch.Generator().manual_seed(SEED59+1)  # 计算并保存当前步骤的中间状态。
    images=torch.zeros(count,1,16,16); images[:,:,4:12,0:4]=1; images[:,:,4:12,12:16]=1  # 计算并保存当前步骤的中间状态。
    images=(images+.03*torch.randn(images.shape,generator=g)).clamp(0,1)  # 计算并保存当前步骤的中间状态。
    masks=torch.zeros(count,16,16); points=torch.zeros(count,3,2); labels=torch.tensor([[1,0,-1]]).repeat(count,1)  # 计算并保存当前步骤的中间状态。
    boxes=torch.zeros(count,1,4)  # 计算并保存当前步骤的中间状态。
    for i in range(count):  # 遍历输入元素以累积或检查结果。
        left=i%2==0; x0,x1=(0,4) if left else (12,16); ox=(13.5 if left else 1.5)  # 计算并保存当前步骤的中间状态。
        masks[i,4:12,x0:x1]=1; points[i,0]=torch.tensor([(x0+x1-1)/2,7.5]); points[i,1]=torch.tensor([float(ox),7.5]); boxes[i,0]=torch.tensor([x0,4,x1-1,11])  # 计算并保存当前步骤的中间状态。
    return images,masks,points,labels,boxes  # 返回当前分支计算出的结果。

images59,masks59,points59,labels59,boxes59=make_prompt_data59()  # 计算并保存当前步骤的中间状态。
box_masks59=torch.ones(boxes59.shape[:2],dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
split59={"train":list(range(24)),"val":list(range(24,28)),"test":list(range(28,32))}  # 计算并保存当前步骤的中间状态。
assert images59.shape==(32,1,16,16) and masks59.sum(1).sum(1).eq(32).all()  # 用受控断言验证关键不变量。
assert box_masks59.shape==(32,1) and box_masks59.all()  # 用受控断言验证关键不变量。
assert torch.equal(images59[0].round(),images59[1].round()) and not torch.equal(masks59[0],masks59[1])  # 用受控断言验证关键不变量。
assert all({i%2 for i in ids}=={0,1} for ids in split59.values())  # 用受控断言验证关键不变量。
regenerated59=make_prompt_data59()  # 计算并保存当前步骤的中间状态。
assert all(torch.equal(a,b) for a,b in zip(regenerated59,(images59,masks59,points59,labels59,boxes59)))  # 用受控断言验证关键不变量。

## 8. 提示分割训练与受控推理

全量小 batch 训练避免 DataLoader 随机性。我们检查 loss 下降、test 最佳候选 IoU，以及在同一图像上把提示从左换成右时输出确实改变。IoU head 的监督来自当前离散候选，早期较噪；正式系统通常采用更精细的多 mask 训练策略与大规模数据引擎。

In [ ]:
torch.manual_seed(SEED59+2); model59=TinyPromptSegmenter59()  # 计算并保存当前步骤的中间状态。
opt59=torch.optim.AdamW(model59.parameters(),lr=4e-3,weight_decay=1e-4); losses59=[]  # 计算并保存当前步骤的中间状态。
tr59=torch.tensor(split59["train"])  # 计算并保存当前步骤的中间状态。
for step59 in range(81):  # 遍历输入元素以累积或检查结果。
    model59.train(); opt59.zero_grad()  # 执行当前语句以推进本节示例。
    logits59,quality59=model59(images59[tr59],points59[tr59],labels59[tr59],boxes59[tr59],box_masks59[tr59])  # 计算并保存当前步骤的中间状态。
    loss59,_=prompt_loss59(logits59,quality59,masks59[tr59]); loss59.backward(); opt59.step(); losses59.append(float(loss59.detach()))  # 计算并保存当前步骤的中间状态。
model59.eval(); te59=torch.tensor(split59["test"])  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    test_logits59,test_quality59=model59(images59[te59],points59[te59],labels59[te59],boxes59[te59],box_masks59[te59])  # 计算并保存当前步骤的中间状态。
    candidate_iou59=binary_iou_targets59(test_logits59,masks59[te59])  # 计算并保存当前步骤的中间状态。
    predicted_best59=test_quality59.argmax(1); true_best59=candidate_iou59.argmax(1)  # 计算并保存当前步骤的中间状态。
    selected59,_=select_best59(test_logits59,test_quality59)  # 计算并保存当前步骤的中间状态。
    selected_iou59=binary_iou_targets59(selected59[:,None],masks59[te59])[:,0]  # 计算并保存当前步骤的中间状态。
    left_logits59,left_q59=model59(images59[28:29],points59[28:29],labels59[28:29],boxes59[28:29],box_masks59[28:29])  # 计算并保存当前步骤的中间状态。
    right_logits59,right_q59=model59(images59[28:29],points59[29:30],labels59[29:30],boxes59[29:30],box_masks59[29:30])  # 计算并保存当前步骤的中间状态。
    left_mask59,_=select_best59(left_logits59,left_q59); right_mask59,_=select_best59(right_logits59,right_q59)  # 计算并保存当前步骤的中间状态。
assert losses59[-1]<losses59[0]*.22 and selected_iou59.mean()>.85  # 用受控断言验证关键不变量。
assert (predicted_best59==true_best59).float().mean()>=.75  # 用受控断言验证关键不变量。
assert (candidate_iou59.max(1).values-candidate_iou59.min(1).values).mean()>.25  # 候选没有坍缩为同一 mask
assert ((left_mask59.sigmoid()>.5)!=(right_mask59.sigmoid()>.5)).sum()>40  # 用受控断言验证关键不变量。
assert torch.isfinite(test_logits59).all() and torch.isfinite(test_quality59).all()  # 用受控断言验证关键不变量。
print({"initial":round(losses59[0],4),"final":round(losses59[-1],4),"controlled_test_iou":round(float(selected_iou59.mean()),4)})  # 执行当前语句以推进本节示例。

## 9. 发布 wrapper 与包外 publisher registry

模型参数摘要逐项绑定 key/dtype/shape/bytes；manifest 同时绑定 image/mask/prompt 数据、split、坐标归一化、label 语义和训练 recipe。包内的 digest 可被攻击者一起重算，所以 loader 必须对照包外只读 registry。返回 wrapper 统一执行 `eval/no_grad` 和最佳候选选择，调用者拿不到“按 GT 选 mask”的捷径。

In [ ]:
def tensor_digest59(t):  # 定义本节可复用的核心函数。
    t=t.detach().cpu().contiguous(); h=hashlib.sha256(); h.update(str(t.dtype).encode()); h.update(canonical_json59(list(t.shape)).encode()); h.update(t.numpy().tobytes()); return h.hexdigest()  # 计算并保存当前步骤的中间状态。
def state_digest59(state):  # 定义本节可复用的核心函数。
    h=hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        t=state[key].detach().cpu().contiguous(); h.update(key.encode()); h.update(str(t.dtype).encode()); h.update(canonical_json59(list(t.shape)).encode()); h.update(t.numpy().tobytes())  # 计算并保存当前步骤的中间状态。
    return h.hexdigest()  # 返回当前分支计算出的结果。
def artifact_digest59(package):  # 定义本节可复用的核心函数。
    signed={"subject":package["subject"],"manifest":package["manifest"],"state_digest":state_digest59(package["state"])}  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(canonical_json59(signed).encode()).hexdigest()  # 返回当前分支计算出的结果。

manifest59={  # 计算并保存当前步骤的中间状态。
    "config":{"dim":24,"heads":4,"num_masks":3},  # 执行当前语句以推进本节示例。
    "data":{name:tensor_digest59(value) for name,value in {"images":images59,"masks":masks59,"points":points59,"labels":labels59,"boxes":boxes59,"box_masks":box_masks59}.items()},  # 执行当前语句以推进本节示例。
    "split":split59,  # 执行当前语句以推进本节示例。
    "preprocess":{"image":"1x16x16-linear-[0,1]","coordinates":"xy/(W-1,H-1)","point_labels":{"positive":1,"negative":0,"padding":-1},"box":"xyxy-inclusive"},  # 执行当前语句以推进本节示例。
    "recipe":{"seed":SEED59+2,"optimizer":"AdamW","steps":81,"lr":4e-3,"threshold":.5,"segmentation_assignment":"min-over-K","candidate_selection":"argmax-predicted-iou"},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
package59={"subject":"tiny-prompt-segmenter59/v1","manifest":copy.deepcopy(manifest59),"state":copy.deepcopy(model59.state_dict())}  # 计算并保存当前步骤的中间状态。
package59["artifact_digest"]=artifact_digest59(package59)  # 计算并保存当前步骤的中间状态。
_PUBLISHER_REGISTRY59=MappingProxyType({package59["subject"]:package59["artifact_digest"]})  # 计算并保存当前步骤的中间状态。

def deep_freeze59(value):  # 定义本节可复用的核心函数。
    if isinstance(value,dict): return MappingProxyType({key:deep_freeze59(item) for key,item in value.items()})  # 按当前条件选择后续控制路径。
    if isinstance(value,(list,tuple)): return tuple(deep_freeze59(item) for item in value)  # 按当前条件选择后续控制路径。
    return value  # 返回当前分支计算出的结果。

class PublishedSegmenter59:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,model,manifest): self._model=model.eval(); self.manifest=deep_freeze59(copy.deepcopy(manifest))  # 定义本节可复用的核心函数。
    @torch.no_grad()  # 为下方定义附加声明式配置。
    def predict(self,images,points,labels,boxes,box_mask=None):  # 定义本节可复用的核心函数。
        logits,quality=self._model(images,points,labels,boxes,box_mask); selected,index=select_best59(logits,quality)  # 计算并保存当前步骤的中间状态。
        return selected.sigmoid()>self.manifest["recipe"]["threshold"],quality,index  # 返回当前分支计算出的结果。

def load_published59(package):  # 定义本节可复用的核心函数。
    actual=artifact_digest59(package)  # 计算并保存当前步骤的中间状态。
    if _PUBLISHER_REGISTRY59.get(package.get("subject"))!=actual: raise ValueError("publisher_digest_mismatch")  # 按当前条件选择后续控制路径。
    if package.get("artifact_digest")!=actual or package["manifest"]!=manifest59: raise ValueError("manifest_mismatch")  # 按当前条件选择后续控制路径。
    loaded=TinyPromptSegmenter59(**package["manifest"]["config"]); loaded.load_state_dict(package["state"],strict=True)  # 计算并保存当前步骤的中间状态。
    return PublishedSegmenter59(loaded,package["manifest"])  # 返回当前分支计算出的结果。

published59=load_published59(package59); pub_mask59,pub_q59,pub_idx59=published59.predict(images59[28:29],points59[28:29],labels59[28:29],boxes59[28:29])  # 计算并保存当前步骤的中间状态。
assert pub_mask59.shape==(1,16,16) and pub_idx59.shape==(1,)  # 用受控断言验证关键不变量。
published_threshold59=published59.manifest["recipe"]["threshold"]  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    published59.manifest["recipe"]["threshold"]=.99  # 计算并保存当前步骤的中间状态。
    raise AssertionError("nested threshold was mutable")  # 遇到非法合同立即显式失败。
except TypeError: pass  # 捕获预期异常并验证失败分支。
assert published59.manifest["recipe"]["threshold"]==published_threshold59==.5  # 用受控断言验证关键不变量。
forged59=copy.deepcopy(package59); forged59["manifest"]["preprocess"]["coordinates"]="y/x swapped"  # 计算并保存当前步骤的中间状态。
key59=next(iter(forged59["state"])); forged59["state"][key59].add_(1); forged59["artifact_digest"]=artifact_digest59(forged59)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_published59(forged59); raise AssertionError("overall re-sign should fail")  # 执行当前语句以推进本节示例。
except ValueError as exc: assert "publisher" in str(exc)  # 捕获预期异常并验证失败分支。

## 10. 复杂度、失败模式与生产差距

- image token 数为 $L$、prompt token 数为 $P$ 时，双向 cross-attention 是 $O(BLPD)$，token self-attention 是 $O(BP^2D)$；高分辨率成本主要仍在 image encoder 与 mask upsampling。
- 失败模式包括 `(x,y)` 写反、box 端点规范不一致、padding 参与 softmax、负点误当 padding、空 prompt 静默通过、用真实 IoU 选候选，以及不同分辨率复用旧坐标。
- 教学版没有官方 SAM 的大规模预训练、强 image encoder、稠密提示、多尺度高分辨率路径、稳定性分数与数据引擎；也没有证明零样本能力。
- 生产要补请求限额、坐标审计、低质量拒识、mask 压缩、批处理、加速后端、监控和真实签名/KMS；本地 registry 只说明信任边界。

## 11. 原始资料

- Kirillov et al., [Segment Anything](https://arxiv.org/abs/2304.02643)
- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- PyTorch 官方文档：[BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)

名称中的“风格”表示复现接口思想和信息流，而非声称参数、训练数据或能力与官方模型一致。